Extract RAW Data From Snowflake

In [104]:
import pandas as pd 
from snowflake.connector import connect 
import warnings
warnings.filterwarnings('ignore')

details = {
    'user': 'NASRULKHAIR',
    'password': 'C@de0123456789',
    'account': 'SSDSCZP-MD39179',
    'warehouse': 'BANK_WH',
    'database': 'BANK_ANALYTICS_DB',
    'schema': 'RAW'
}

conn = connect(**details)

query = 'SELECT * FROM BANK_CHURN'
df = pd.read_sql(query, conn)
df.head()

,ROWNUMBER,CUSTOMERID,CREDITSCORE,GEOGRAPHYID,GENDERID,AGE,TENURE,BALANCE,NUMOFPRODUCTS,HASCRCARD,ISACTIVEMEMBER,ESTIMATEDSALARY,EXITED,BANK_DOJ
0,1,15634602,619,1,2,42,7,0.00,1,1,1,101348.88,1,2016-03-23
1,2,15647311,608,2,2,41,4,83807.86,1,0,1,112542.58,0,2018-10-09
2,3,15619304,502,1,2,42,4,159660.80,3,1,0,113931.57,1,2019-03-25
3,4,15701354,699,1,2,39,3,0.00,2,0,0,93826.63,0,2019-10-24
4,5,15737888,850,2,2,43,3,125510.82,1,1,1,79084.10,0,2019-11-22


##### Data Cleaning


In [105]:
df.isnull().sum()

ROWNUMBER          0
CUSTOMERID         0
CREDITSCORE        0
GEOGRAPHYID        0
GENDERID           0
AGE                0
TENURE             0
BALANCE            0
NUMOFPRODUCTS      0
HASCRCARD          0
ISACTIVEMEMBER     0
ESTIMATEDSALARY    0
EXITED             0
BANK_DOJ           0
dtype: int64

In [106]:
df.columns

Index(['ROWNUMBER', 'CUSTOMERID', 'CREDITSCORE', 'GEOGRAPHYID', 'GENDERID',
       'AGE', 'TENURE', 'BALANCE', 'NUMOFPRODUCTS', 'HASCRCARD',
       'ISACTIVEMEMBER', 'ESTIMATEDSALARY', 'EXITED', 'BANK_DOJ'],
      dtype='object')

In [107]:
# renaming column name

df = df.rename(columns={
    'ROWNUMBER':'row_number', 
    'CUSTOMERID':'customer_id', 
    'CREDITSCORE':'credit_score', 
    'GEOGRAPHYID': 'geography_id', 
    'GENDERID': 'gender_id',
    'AGE': 'age', 
    'TENURE': 'tenure',
    'BALANCE': 'balance',
    'NUMOFPRODUCTS': 'num_of_products',
    'HASCRCARD': 'has_cr_card',
    'ISACTIVEMEMBER': 'is_active_member',
    'ESTIMATEDSALARY': 'estimated_salary',
    'EXITED': 'exited',
    'BANK_DOJ': 'bank_doj'
})

df.head()

,row_number,customer_id,credit_score,geography_id,gender_id,age,tenure,balance,num_of_products,has_cr_card,is_active_member,estimated_salary,exited,bank_doj
0,1,15634602,619,1,2,42,7,0.00,1,1,1,101348.88,1,2016-03-23
1,2,15647311,608,2,2,41,4,83807.86,1,0,1,112542.58,0,2018-10-09
2,3,15619304,502,1,2,42,4,159660.80,3,1,0,113931.57,1,2019-03-25
3,4,15701354,699,1,2,39,3,0.00,2,0,0,93826.63,0,2019-10-24
4,5,15737888,850,2,2,43,3,125510.82,1,1,1,79084.10,0,2019-11-22


In [108]:
# checking for data types
#print(df.dtypes)

# changing bank_doj to datetime format
df['bank_doj'] = pd.to_datetime(df['bank_doj'])
print(df.dtypes)


row_number                   int64
customer_id                  int64
credit_score                 int64
geography_id                 int64
gender_id                    int64
age                          int64
tenure                       int64
balance                    float64
num_of_products              int64
has_cr_card                  int64
is_active_member             int64
estimated_salary           float64
exited                       int64
bank_doj            datetime64[ns]
dtype: object


In [109]:
# map geography_id to country names

df['geography_name'] = df['geography_id'].map({1: 'France', 2: 'Spain', 3: 'Germany'})
df['geography_name'].value_counts()


geography_name
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [110]:
# map gender_id to gender

df['gender'] = df['gender_id'].map({1:'Male',2: 'Female'})
df['gender'].value_counts()

gender
Male      5457
Female    4543
Name: count, dtype: int64

In [111]:
# map exited to exit and retain

df['churned'] = df['exited'].map({1:'Exit', 0: 'Retain'})
df['churned'].head()



0      Exit
1    Retain
2      Exit
3    Retain
4    Retain
Name: churned, dtype: object

In [112]:
# map active_id to active and non active members
df['active_member'] = df['is_active_member'].map({1:'Active Member', 0: 'Non Active Member'})
df['active_member'].value_counts()

active_member
Active Member        5151
Non Active Member    4849
Name: count, dtype: int64

In [113]:
# mapr has_cr_card to credit card holder and non credit card holder

df['credit_card_holder'] = df['has_cr_card'].map({1:'Credit Card Holder', 0: 'Non Credit Card Holder'})
df['credit_card_holder'].value_counts()

credit_card_holder
Credit Card Holder        7055
Non Credit Card Holder    2945
Name: count, dtype: int64

In [114]:
df.columns

Index(['row_number', 'customer_id', 'credit_score', 'geography_id',
       'gender_id', 'age', 'tenure', 'balance', 'num_of_products',
       'has_cr_card', 'is_active_member', 'estimated_salary', 'exited',
       'bank_doj', 'geography_name', 'gender', 'churned', 'active_member',
       'credit_card_holder'],
      dtype='object')

In [115]:
# Rearranging columns
df = df[['row_number', 'customer_id', 'credit_score', 'geography_id', 'geography_name',
       'gender_id', 'gender', 'age', 'tenure', 'balance', 'num_of_products',
       'has_cr_card', 'credit_card_holder', 'is_active_member', 'active_member', 'estimated_salary', 'exited', 'churned',
       'bank_doj' ]]

df.head()

,row_number,customer_id,credit_score,geography_id,geography_name,gender_id,gender,age,tenure,balance,num_of_products,has_cr_card,credit_card_holder,is_active_member,active_member,estimated_salary,exited,churned,bank_doj
0,1,15634602,619,1,France,2,Female,42,7,0.00,1,1,Credit Card Holder,1,Active Member,101348.88,1,Exit,2016-03-23
1,2,15647311,608,2,Spain,2,Female,41,4,83807.86,1,0,Non Credit Card Holder,1,Active Member,112542.58,0,Retain,2018-10-09
2,3,15619304,502,1,France,2,Female,42,4,159660.80,3,1,Credit Card Holder,0,Non Active Member,113931.57,1,Exit,2019-03-25
3,4,15701354,699,1,France,2,Female,39,3,0.00,2,0,Non Credit Card Holder,0,Non Active Member,93826.63,0,Retain,2019-10-24
4,5,15737888,850,2,Spain,2,Female,43,3,125510.82,1,1,Credit Card Holder,1,Active Member,79084.10,0,Retain,2019-11-22


##### Final Cleaning


In [116]:
# dropping unwanted columns
df = df.drop(columns=['row_number'])
df.head()

,customer_id,credit_score,geography_id,geography_name,gender_id,gender,age,tenure,balance,num_of_products,has_cr_card,credit_card_holder,is_active_member,active_member,estimated_salary,exited,churned,bank_doj
0,15634602,619,1,France,2,Female,42,7,0.00,1,1,Credit Card Holder,1,Active Member,101348.88,1,Exit,2016-03-23
1,15647311,608,2,Spain,2,Female,41,4,83807.86,1,0,Non Credit Card Holder,1,Active Member,112542.58,0,Retain,2018-10-09
2,15619304,502,1,France,2,Female,42,4,159660.80,3,1,Credit Card Holder,0,Non Active Member,113931.57,1,Exit,2019-03-25
3,15701354,699,1,France,2,Female,39,3,0.00,2,0,Non Credit Card Holder,0,Non Active Member,93826.63,0,Retain,2019-10-24
4,15737888,850,2,Spain,2,Female,43,3,125510.82,1,1,Credit Card Holder,1,Active Member,79084.10,0,Retain,2019-11-22


In [117]:
# Renaming column for data modelling

df.rename(columns = {
    'is_active_member': 'active_id',
    'has_cr_card': 'cr_card_id',
    'exited': 'exit_id',
    'is_active_member': 'active_id',
    'geography_name': 'country_name'
}, inplace=True)

df.head()

,customer_id,credit_score,geography_id,country_name,gender_id,gender,age,tenure,balance,num_of_products,cr_card_id,credit_card_holder,active_id,active_member,estimated_salary,exit_id,churned,bank_doj
0,15634602,619,1,France,2,Female,42,7,0.00,1,1,Credit Card Holder,1,Active Member,101348.88,1,Exit,2016-03-23
1,15647311,608,2,Spain,2,Female,41,4,83807.86,1,0,Non Credit Card Holder,1,Active Member,112542.58,0,Retain,2018-10-09
2,15619304,502,1,France,2,Female,42,4,159660.80,3,1,Credit Card Holder,0,Non Active Member,113931.57,1,Exit,2019-03-25
3,15701354,699,1,France,2,Female,39,3,0.00,2,0,Non Credit Card Holder,0,Non Active Member,93826.63,0,Retain,2019-10-24
4,15737888,850,2,Spain,2,Female,43,3,125510.82,1,1,Credit Card Holder,1,Active Member,79084.10,0,Retain,2019-11-22


### Pushing Clean Data ito clean schema in snowflake


In [118]:
from snowflake.connector.pandas_tools import write_pandas

# Make sure the CLEAN schema exists
conn.cursor().execute("USE SCHEMA BANK_ANALYTICS_DB.CLEAN")

# Write dataframe to Snowflake
success, nchunks, nrows, _ = write_pandas(
    conn=conn,
    df=df,
    table_name='BANK_CHURN_CLEANED',
    schema='CLEAN',  # target schema
    auto_create_table=True,  # create if doesn't exist
    overwrite=True  # replaces existing table
)

print(f"Upload successful: {success}, rows inserted: {nrows}")


Upload successful: True, rows inserted: 10000
